# GNN — 2 Lepton 2 Taus (Run 2 + Run 3 + Combined)

Graph-neural-network counterpart to `DNN_2l2tau.ipynb`'s flat-feature MLP,
on the same 2l2tau data (`Evie/PPSSP_2026/2l2tau/run2` and `.../run3`), same
preselection, same leakage-free feature policy. Cross-validation here
follows `MLP_2l2tau.ipynb`'s (and the 1l2tau `GNN_Evie_final_1l2t.ipynb`'s)
5-fold `eventNumber % N_FOLDS` convention rather than a single fixed
train/val/test split - `DNN_2l2tau.ipynb` has not been refactored to that
convention yet, so its own numbers are not directly comparable to this
notebook's OOF results.

**Representation change.** The MLP concatenates every event into one flat
feature vector (kinematics + hand-engineered pairwise variables like `dR_*`,
`m_*`, `HT_*`). This notebook instead represents each event as a small graph:
one node per reconstructed physics object (lepton1, lepton2, tau1, tau2,
jet1, jet2, MET), each carrying only its own kinematics, with a
fully-connected edge set within the event. The hand-engineered
pairwise/aggregate variables are deliberately **not** given to the model -
the point of the GNN is to see how much of that relational information
(angular separations, invariant masses, combined momenta) message passing
can recover on its own from raw 4-vectors.

**Node count vs 1l2tau.** This channel has one extra node (lepton2) instead
of a single lepton - 7 nodes/event instead of 6.

**Jets aren't guaranteed here.** Unlike 1l2tau (`n_jet >= 2` in the
preselection), 2l2tau's preselection does not require any jets - `j1_*`/
`j2_*` are frequently missing (sentinel-masked to NaN by `clean_data`, then
median-imputed same as any other missing value, independently per fold).
An imputed "typical jet" for an event with no real jet is a modelling
compromise worth flagging in your write-up, not a bug - a `has_jet1`/
`has_jet2` indicator feature would be a reasonable follow-up if you want
the model to distinguish real vs imputed jets explicitly.

**A caveat on the node schema below.** `DNN_2l2tau.ipynb` builds its
feature list dynamically (`discover_common_features`), so I could only
confirm a subset of raw per-object branch names from what its feature-
importance/correlation cells happened to print (`l1_e`, `l1_charge`,
`l2_e`, `l2_charge`, `j1_pt`, `j2_pt`, `j2_eta`, `tau1_pt`, `tau2_pt`,
`met_met`, `met_sumet`). Columns like `l1_eta`/`l1_phi`/`l2_eta`/`l2_phi`/
`l1_pdg`/`l2_pdg`/`j1_eta`/`j1_phi`/`j2_phi`/`met_phi`/`tau1_eta`/
`tau1_phi`/`tau2_eta`/`tau2_phi` are assumed present by analogy with the
1l2tau ntuples (same production, same tree), but **not individually
verified** - the schema cell below has a runtime `assert` that will name
the exact missing column if any of these guesses are wrong, so a failure
there is expected to be a quick one-line fix, not a sign anything else is
broken.

**Scope.** Mirrors `GNN_Evie_final_1l2t.ipynb`'s k-fold structure exactly:
data loading, graph construction, model, training with early stopping,
ROC/AUC evaluation, and the same 5-fold out-of-fold (OOF) production run
(`run_kfold_gnn`) - for Run 2 first, then Run 3 as a separate downstream
section, then Combined Run2+Run3 as a third section (same `_run3`/
`_comb`-suffixed variable convention, all three reusing the Run 2 section's
functions/classes/schema unchanged, including the K-Fold + weight-balancing
helper library). Permutation importance is adapted below (ranking the raw
per-object columns instead of ~80 flat features) for all three; correlation
pruning and the hyperparameter grid search are left out to keep this
notebook focused - see `DNN_2l2tau.ipynb`/`MLP_2l2tau.ipynb` for those.


## Libraries

In [ ]:
import os

# Must be set BEFORE CUDA/cuBLAS initializes for deterministic cuBLAS matmul.
# If this kernel already has CUDA initialized (e.g. you've run cells before
# adding this), these two env vars won't take effect until you RESTART THE
# KERNEL.

os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# torch and uproot (via numpy) can each bundle their own OpenMP runtime;
# loading both in one process aborts the kernel on some platforms
# ("OMP: Error #15 ... libomp.dylib already initialized") unless this is
# set before either is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

RANDOM_STATE = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = RANDOM_STATE):

    """
    Full determinism, not just seeding: also pins cuDNN to deterministic
    kernels, disables its autotuner, and asks torch to error out (rather than
    silently fall back) on any op without a deterministic implementation.
    Same convention as DNN.ipynb's set_seed, so re-seeded runs here are
    reproducible the same way. Determinism is only guaranteed on the SAME
    machine / CUDA / torch version - it is not portable across hardware or
    library versions.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


set_seed(RANDOM_STATE)

print(f"Using device: {DEVICE}")

In [ ]:
# ---------------------------------------------------------------------------
# Paths - 2 leptons + 2 taus, Run 2 and Run 3 (same source files as
# DNN_2l2tau.ipynb / Final_Notebooks/2L2Tau_Master_Pipeline.ipynb).
# ---------------------------------------------------------------------------

BASE_DIR_RUN2 = Path("PPSSP_2026/2l2tau/run2")
BASE_DIR_RUN3 = Path("PPSSP_2026/2l2tau/run3")

# Active run used by downstream cells by default (Run 2 first pass below;
# switched to Run 3 in the separate downstream section further down).
BASE_DIR = BASE_DIR_RUN2
ACTIVE_RUN = "Run 2"
TREE_NAME = "AnalysisMiniTree"

# ---------------------------------------------------------------------------
# Preselection (see repo README.md, "2 Lepton 2 Taus")
# ---------------------------------------------------------------------------

PRESELECTION = "(n_b_jet == 0) & (l1_charge * l2_charge < 0) & (mZ_cut > 0)"

# ---------------------------------------------------------------------------
# Processes: filename + label (1 = signal, 0 = background)
# ---------------------------------------------------------------------------

FILES = {
    "signal_ggF": ("signal_ggF.root", 1),
    "signal_VBF": ("signal_VBF.root", 1),
    "Diboson":    ("diboson.root",    0),
    "Zjets":      ("Zjets.root",      0),
    "Wjets":      ("Wjets.root",      0),
    "ttbar":      ("ttbar.root",      0),
    "tops":       ("tops.root",       0),
    "SingleH":    ("singleH.root",    0),
    "Vgamma":     ("Vgamma.root",     0),
    "VVV":        ("VVV.root",        0),
}

WEIGHT_PARTS = ["weight", "weights"]  # raw branches; w_phys = their product
EVENT_ID_BRANCH = "eventNumber"        # bookkeeping only - NEVER a feature (see BLOCK_SUBSTR)

# ---------------------------------------------------------------------------
# Leakage-free feature-selection policy - IDENTICAL to DNN_2l2tau.ipynb, so
# this notebook builds graph nodes only out of branches the MLP was also
# allowed to see.
# ---------------------------------------------------------------------------

BLOCK_SUBSTR = ["weight", "effsf", "_ff", "truth", "istrue", "fake", "anti",
                "dsid", "eventnumber", "_RNNTight", "_isOS", "_d0sig"]
BLOCK_EXACT = {
    "n_b_jet", "pass2l2tau", "tau1_RNNJetScoreSigTrans", "tau2_RNNJetScoreSigTrans",
    "pair_isOStaus", "pair_isOSlep1lep2", "tau2_baseline_RNNTight", "l1_d0sig",
    "tau1_charge", "tau2_charge", "mZ_veto", "tau1_decayMode", "tau2_decayMode",
    "tau1_nprong", "tau2_nprong", "mZ_cut", "mZreq",
}

BLOCK_EXACT_LOWER = {b.lower() for b in BLOCK_EXACT}


def is_feature(branch: str) -> bool:

    """True if `branch` is safe to use as a training feature (see policy above)."""

    lb = branch.lower()
    return lb not in BLOCK_EXACT_LOWER and not any(s.lower() in lb for s in BLOCK_SUBSTR)


# ---------------------------------------------------------------------------
# K-fold knobs - shared with every other refactored notebook in this project
# (see assign_folds below). N_OPTUNA_INNER_FOLDS is smaller than N_FOLDS
# deliberately: NN training is far more expensive per-fold than XGBoost's,
# so the Optuna search itself uses fewer folds than the final OOF rotation
# (which always uses the full N_FOLDS) - an efficiency tradeoff, not a
# methodology difference.
# ---------------------------------------------------------------------------

N_FOLDS = 5                # outer K-fold CV - fold = eventNumber % N_FOLDS
N_OPTUNA_TRIALS = 25
N_OPTUNA_INNER_FOLDS = 2   # folds used INSIDE the Optuna search only

## Data Loading Helpers

Identical to `DNN_2l2tau.ipynb`'s loaders - same discovery, same
preselection, same sentinel-to-NaN cleaning, and `eventNumber` is now read
alongside the features (never used as one - see `BLOCK_SUBSTR`/
`EVENT_ID_BRANCH`) so `assign_folds` below can build a deterministic fold
assignment, same convention as `MLP_2l2tau.ipynb`'s Run 2 `data`/
`features`.

In [ ]:
def discover_common_features(base_dir, files=FILES, tree_name=TREE_NAME):

    """
    Branches common to EVERY process file in `base_dir`, filtered through`is_feature`.
    """

    common = None

    for fname, _ in files.values():
        keys = set(uproot.open({str(Path(base_dir) / fname): tree_name}).keys())
        common = keys if common is None else common & keys

    features = sorted(b for b in common if is_feature(b))

    print(f"{len(features)} candidate features (common to all {len(files)} processes, leakage-free)")

    return features


def load_run_data(base_dir, features, files=FILES, weight_parts=WEIGHT_PARTS,
                   preselection=PRESELECTION, tree_name=TREE_NAME,
                   event_id_branch=EVENT_ID_BRANCH, verbose=True):

    """
    Read every process file under `base_dir`, apply the preselection at
    read time, and concatenate into one DataFrame with bookkeeping columns:
      - w_phys  : physical event weight = weight * weights
      - label   : 1 = signal, 0 = background
      - process : originating process name
      - eventNumber : read alongside the features (never used as one - see
        BLOCK_SUBSTR/EVENT_ID_BRANCH) so `assign_folds` can build a
        deterministic fold assignment downstream.
    """

    base_dir = Path(base_dir)
    dfs = []

    for proc, (fname, label) in files.items():

        tree = uproot.open({str(base_dir / fname): tree_name})
        df = tree.arrays(features + weight_parts + [event_id_branch], cut=preselection, library="pd")
        df["w_phys"] =  df["weights"] * df["weight"]
        df["label"] = label
        df["process"] = proc
        dfs.append(df)

        if verbose:
            print(f"{proc:12s}: {len(df):>8d} events after preselection")

    return pd.concat(dfs, ignore_index=True)


def clean_data(data, features, verbose=True):

    """
    Post-concat cleaning: drop constant/empty features, then mask sentinel
    values (< -100, e.g. -999) to NaN. Returns (cleaned_data, updated_features);
    operates on a copy, does not mutate the input DataFrame.
    """

    data = data.copy()
    nun = data[features].nunique()
    const = nun[nun <= 1].index.tolist()
    features = [f for f in features if f not in const]
    data = data.drop(columns=const)

    if verbose:
        print(f"Dropped {len(const)} constant/empty features:\n  {sorted(const)}")

    for f in features:
        m = data[f] < -100
        if m.any():
            if verbose:
                print(f"  sentinel -> NaN: {f} ({m.mean():.1%})")
            data[f] = data[f].mask(m)

    if verbose:
        print(f"\n{len(features)} final features")
        print(f"Total: {len(data)} events | signal = {(data.label==1).sum()} | "
              f"background = {(data.label==0).sum()}")
        print(f"Yield (w_phys): signal = {data.loc[data.label==1,'w_phys'].sum():.2f} | "
              f"background = {data.loc[data.label==0,'w_phys'].sum():.2f}")
    return data, features

## K-Fold + Weight-Balancing Helper Library

Same 5-fold `eventNumber % N_FOLDS` convention, weight handling and
significance objective (`significance_scan`) as the other refactored NN
notebooks in this project (`DNN.ipynb`, `MLP.ipynb`, `PNN.ipynb`).
`make_fit_weights` is copy-pasted BYTE-FOR-BYTE from the canonical source
(`PNN.ipynb`) - unchanged here, same as every other in-scope notebook.
`prepare_fold_tensors_gnn`/`run_kfold_gnn` are the GNN-specific analogues of
`DNN.ipynb`'s generic `prepare_fold_tensors`/`run_kfold_nn`: instead of a
flat feature vector + `__isnan` flags, they repeat this notebook's own
per-fold median-impute + `StandardScaler` (continuous node columns only) +
`stack_node_features`/`build_graph_tensors` pipeline - the node schema
(`REQUIRED_OBJECT_COLUMNS`/`CONTINUOUS_NODE_COLS`) is fixed, so unlike the
flat-vector NN notebooks there is no per-fold feature selection to carry
through, and there is no Optuna search in this notebook so `run_kfold_gnn`
trains every fold with the same frozen default hyperparameters. Everything
else (`assign_folds`, `make_fit_weights`, `significance_scan`,
`prepare_fold_data`) is channel/model-agnostic and identical to the other
notebooks' own K-Fold library.

In [ ]:
def assign_folds(df, event_col=EVENT_ID_BRANCH, n_folds=N_FOLDS):

    """Deterministic fold assignment: fold = eventNumber % n_folds. The SAME
    rule is applied identically across every notebook/channel/run/track in
    this project, so OOF score arrays stay event-aligned everywhere."""

    df = df.copy()
    df["fold"] = (df[event_col].to_numpy() % n_folds).astype("int8")
    return df


def compute_process_yield_targets(df, weight_col="w_phys", process_col="process"):

    """Full-sample SIGNED yield per process, computed ONCE on the complete
    dataset (before any negative-weight drop)."""

    return df.groupby(process_col)[weight_col].sum().to_dict()


def rescale_weights_by_yield(df, target_yields, weight_col="w_phys",
                              process_col="process", min_target=1e-6):

    """Per-process: rescale the rows in `df` (already negative-weight
    filtered) so their weight-sum matches `target_yields[process]`."""

    df = df.copy()
    for proc, sub in df.groupby(process_col):
        target = target_yields.get(proc, sub[weight_col].sum())
        if target <= min_target:
            warnings.warn(
                f"rescale_weights_by_yield: process '{proc}' has full-sample "
                f"signed yield {target:.6g} <= {min_target:g} (mostly-negative "
                f"weights?) - clamping target to {min_target:g} to avoid a "
                f"negative/degenerate rescale factor."
            )
            target = min_target
        kept_sum = sub[weight_col].sum()
        if kept_sum != 0:
            df.loc[sub.index, weight_col] = sub[weight_col] * (target / kept_sum)
    return df


def make_fit_weights(labels, abs_weights, cell_ids=None):

    """Balance signal/background total weight and normalize the OVERALL
    mean weight to 1. If `cell_ids` is given, the balance is computed
    WITHIN each distinct cell_ids value FIRST.

    CANONICAL SOURCE: PNN.ipynb. Copy-pasted BYTE-FOR-BYTE into every
    in-scope notebook - see assert_fit_weights_balanced below.
    """

    labels = np.asarray(labels)
    fit_weights = np.asarray(abs_weights, dtype=float).copy()

    if cell_ids is None:
        sum_signal = fit_weights[labels == 1].sum()
        sum_background = fit_weights[labels == 0].sum()
        fit_weights[labels == 1] *= sum_background / sum_signal
    else:
        cell_ids = np.asarray(cell_ids)
        for cell in np.unique(cell_ids):
            m = cell_ids == cell
            sum_signal = fit_weights[m & (labels == 1)].sum()
            sum_background = fit_weights[m & (labels == 0)].sum()
            if sum_signal > 0:
                fit_weights[m & (labels == 1)] *= sum_background / sum_signal

    fit_weights *= len(fit_weights) / fit_weights.sum()

    return fit_weights


def assert_fit_weights_balanced(fit_weights, labels, cell_ids):

    """Per-cell balance sanity check for make_fit_weights."""

    labels = np.asarray(labels)
    cell_ids = np.asarray(cell_ids)
    cells = np.unique(cell_ids)
    sig_sums = [fit_weights[(labels == 1) & (cell_ids == c)].sum() for c in cells]
    bkg_sums = [fit_weights[(labels == 0) & (cell_ids == c)].sum() for c in cells]
    assert np.allclose(sig_sums, bkg_sums), (
        "make_fit_weights: per-cell balance broken - this copy has drifted "
        "from the canonical PNN.ipynb version"
    )


def n_eff_table(df, group_cols, weight_col="w_phys"):

    """Effective event count N_eff = (sum w)^2 / sum(w^2) per group."""

    def _n_eff(w):
        w = np.asarray(w, dtype=float)
        denom = (w ** 2).sum()
        if denom == 0:
            return np.nan
        return (w.sum()) ** 2 / denom

    return df.groupby(list(group_cols))[weight_col].apply(_n_eff).rename("n_eff")


def plot_weight_balance(y, w_before, w_after, title="", save_path=None):

    """Panel A: overlaid per-event weight histograms (signal vs background),
    before vs after `make_fit_weights` balancing, log-y. Panel B: grouped
    bar chart of summed weight (signal total vs background total), before
    vs after."""

    y = np.asarray(y)
    w_before = np.asarray(w_before, dtype=float)
    w_after = np.asarray(w_after, dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    lo = min(w_before.min(), w_after.min())
    hi = max(w_before.max(), w_after.max())
    bins = np.linspace(lo, hi, 60)
    ax.hist(w_before[y == 1], bins=bins, histtype="step", linestyle="--",
            color="crimson", linewidth=1.6, alpha=0.7, label="signal (before)")
    ax.hist(w_before[y == 0], bins=bins, histtype="step", linestyle="--",
            color="steelblue", linewidth=1.6, alpha=0.7, label="background (before)")
    ax.hist(w_after[y == 1], bins=bins, histtype="step", linestyle="-",
            color="crimson", linewidth=1.8, label="signal (after)")
    ax.hist(w_after[y == 0], bins=bins, histtype="step", linestyle="-",
            color="steelblue", linewidth=1.8, label="background (after)")
    ax.set_yscale("log")
    ax.set_xlabel("per-event weight")
    ax.set_ylabel("events (log scale)")
    ax.set_title("Per-event weight distribution")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[1]
    sums_before = [w_before[y == 1].sum(), w_before[y == 0].sum()]
    sums_after = [w_after[y == 1].sum(), w_after[y == 0].sum()]
    x = np.arange(2)
    width = 0.35
    b1 = ax.bar(x - width / 2, sums_before, width, label="before", color="lightgray", edgecolor="black")
    b2 = ax.bar(x + width / 2, sums_after, width, label="after", color="steelblue")
    ax.bar_label(b1, fmt="%.3g", fontsize=8)
    ax.bar_label(b2, fmt="%.3g", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(["signal", "background"])
    ax.set_ylabel("summed weight")
    ax.set_title("Total weight: signal vs background")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    fig.suptitle(title)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        print(f"Saved plot -> {save_path}")
    plt.show()


def significance_scan(y_true, scores, w_phys, n_thr=200, min_bkg=1.0):

    """Max Asimov significance Z over score cuts. Uses SIGNED w_phys. This
    is the canonical significance objective for every model in this
    project."""

    thr = np.quantile(scores, np.linspace(0, 1, n_thr))
    best_z, best_t = 0.0, None
    for t in thr:
        sel = scores >= t
        S = w_phys[sel & (y_true == 1)].sum()
        B = w_phys[sel & (y_true == 0)].sum()
        if S <= 0 or B < min_bkg:
            continue
        z = np.sqrt(2 * ((S + B) * np.log(1 + S / B) - S))
        if z > best_z:
            best_z, best_t = z, t
    return best_z, best_t


def prepare_fold_data(data, features, target_yields, cell_cols=(), n_folds=N_FOLDS, k=0):

    """Build the train/val/test row partitions and per-row training weights
    for outer-fold rotation `k`: test=(fold==k), val=(fold==(k+1)%n_folds),
    train=the remaining n_folds-2 folds."""

    val_fold = (k + 1) % n_folds
    test_mask = data["fold"] == k
    val_mask = data["fold"] == val_fold
    train_mask = ~(test_mask | val_mask)

    train_full = data.loc[train_mask]
    val_full = data.loc[val_mask]

    train_keep = train_full.loc[train_full["w_phys"] >= 0].copy()
    val_keep = val_full.loc[val_full["w_phys"] >= 0].copy()

    train_keep = rescale_weights_by_yield(train_keep, target_yields)
    val_keep = rescale_weights_by_yield(val_keep, target_yields)

    def _cell_ids(df):
        if not cell_cols:
            return None
        return df[list(cell_cols)].astype(str).agg("_".join, axis=1).to_numpy()

    cell_train = _cell_ids(train_keep)
    w_train_fit = make_fit_weights(train_keep["label"].to_numpy(), train_keep["w_phys"].to_numpy(), cell_ids=cell_train)
    if cell_train is not None:
        assert_fit_weights_balanced(w_train_fit, train_keep["label"].to_numpy(), cell_train)

    cell_val = _cell_ids(val_keep)
    w_val_fit = make_fit_weights(val_keep["label"].to_numpy(), val_keep["w_phys"].to_numpy(), cell_ids=cell_val)

    return {
        "k": k, "val_fold": val_fold,
        "train_df": train_keep, "val_df": val_keep, "test_df": data.loc[test_mask].copy(),
        "w_train_fit": w_train_fit, "w_val_fit": w_val_fit,
        "n_dropped_train": len(train_full) - len(train_keep),
        "n_dropped_val": len(val_full) - len(val_keep),
    }


def prepare_fold_tensors_gnn(data, target_yields, cell_cols=(), n_folds=N_FOLDS, k=0):

    """
    GNN analogue of DNN.ipynb's prepare_fold_tensors: builds the
    train/val/test row partitions and per-row training weights via
    prepare_fold_data, then repeats this notebook's own per-fold median-
    impute + StandardScaler (continuous node columns only, fit on THAT
    FOLD's train split) + stack_node_features/build_graph_tensors pipeline.
    The node schema (REQUIRED_OBJECT_COLUMNS/CONTINUOUS_NODE_COLS, defined
    in the Object/Node Schema section below) is fixed - unlike the
    flat-vector NN notebooks there is no per-fold feature selection or
    __isnan flags to carry through.
    """

    fd = prepare_fold_data(data, REQUIRED_OBJECT_COLUMNS, target_yields,
                            cell_cols=cell_cols, n_folds=n_folds, k=k)
    train_df, val_df, test_df = fd["train_df"], fd["val_df"], fd["test_df"]

    train_medians = train_df[REQUIRED_OBJECT_COLUMNS].median()
    train_imp = train_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)
    val_imp = val_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)
    test_imp = test_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)

    scaler = StandardScaler()
    train_scaled = train_imp.copy()
    val_scaled = val_imp.copy()
    test_scaled = test_imp.copy()
    train_scaled[CONTINUOUS_NODE_COLS] = scaler.fit_transform(train_imp[CONTINUOUS_NODE_COLS])
    val_scaled[CONTINUOUS_NODE_COLS] = scaler.transform(val_imp[CONTINUOUS_NODE_COLS])
    test_scaled[CONTINUOUS_NODE_COLS] = scaler.transform(test_imp[CONTINUOUS_NODE_COLS])

    assert np.isfinite(train_scaled.to_numpy()).all(), "NaN/inf reached the model input (train)"
    assert np.isfinite(val_scaled.to_numpy()).all(), "NaN/inf reached the model input (val)"
    assert np.isfinite(test_scaled.to_numpy()).all(), "NaN/inf reached the model input (test)"

    X_train_t, y_train_t, w_train_fit_t, w_train_abs_t = build_graph_tensors(
        train_scaled, train_imp, train_df, fit_weights=fd["w_train_fit"])
    X_val_t, y_val_t, _, w_val_abs_t = build_graph_tensors(val_scaled, val_imp, val_df)
    X_test_t, y_test_t, _, w_test_abs_t = build_graph_tensors(test_scaled, test_imp, test_df)

    fd["train_scaled"], fd["train_imp"] = train_scaled, train_imp
    fd["val_scaled"], fd["val_imp"] = val_scaled, val_imp
    fd["test_scaled"], fd["test_imp"] = test_scaled, test_imp
    fd["X_train_t"], fd["y_train_t"], fd["w_train_fit_t"], fd["w_train_abs_t"] = X_train_t, y_train_t, w_train_fit_t, w_train_abs_t
    fd["X_val_t"], fd["y_val_t"], fd["w_val_abs_t"] = X_val_t, y_val_t, w_val_abs_t
    fd["X_test_t"], fd["y_test_t"], fd["w_test_abs_t"] = X_test_t, y_test_t, w_test_abs_t
    fd["scaler"], fd["train_medians"] = scaler, train_medians

    return fd


def run_kfold_gnn(data, target_yields, cell_cols=(), n_folds=N_FOLDS, label="",
                   hidden_channels=None, dropout=None, lr=None, batch_size=None,
                   n_epochs=None, patience=None):

    """
    GNN analogue of DNN.ipynb's run_kfold_nn: 5-fold rotation, one
    ObjectGNN per outer fold (frozen tuned hyperparameters - see the Optuna
    Hyperparameter Search section above), scoring every event with a model
    that never trained on it. Returns (oof_df, models, scalers,
    medians_list).
    """

    hidden_channels = DEFAULT_HIDDEN_CHANNELS if hidden_channels is None else hidden_channels
    dropout = DEFAULT_DROPOUT if dropout is None else dropout
    lr = LEARNING_RATE if lr is None else lr
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    n_epochs = N_EPOCHS if n_epochs is None else n_epochs
    patience = PATIENCE if patience is None else patience

    oof_scores = np.full(len(data), np.nan)
    models, scalers, medians_list = [], [], []

    for k in range(n_folds):
        fd = prepare_fold_tensors_gnn(data, target_yields, cell_cols=cell_cols, n_folds=n_folds, k=k)

        fold_model, _, _, _, _ = train_model(
            fd["X_train_t"], fd["y_train_t"], fd["w_train_fit_t"], fd["w_train_abs_t"],
            fd["X_val_t"], fd["y_val_t"], fd["w_val_abs_t"],
            hidden_channels=hidden_channels, dropout=dropout, lr=lr, batch_size=batch_size,
            n_epochs=n_epochs, patience=patience, verbose=False,
        )
        models.append(fold_model)
        scalers.append(fd["scaler"])
        medians_list.append(fd["train_medians"])

        test_scores = score_dataset(fold_model, fd["X_test_t"], batch_size=batch_size).cpu().numpy()
        test_df = fd["test_df"]
        oof_scores[np.flatnonzero((data["fold"] == k).to_numpy())] = test_scores

        auc_test = roc_auc_score(test_df["label"], test_scores, sample_weight=test_df["w_phys"].abs())
        z_test, _ = significance_scan(test_df["label"].to_numpy(), test_scores, test_df["w_phys"].to_numpy())
        print(f"[{label}] fold {k}: test_fold={k} val_fold={fd['val_fold']} | "
              f"train n={len(fd['train_df'])} (dropped {fd['n_dropped_train']} neg) | "
              f"test weighted AUC = {auc_test:.4f} | test significance Z = {z_test:.3f}")

    oof_cols = ["process", "label", "fold", EVENT_ID_BRANCH, "w_phys"]
    if "run" in data.columns:
        oof_cols.append("run")
    oof_df = data[oof_cols].copy()
    oof_df["score"] = oof_scores

    oof_auc = roc_auc_score(oof_df["label"], oof_df["score"], sample_weight=oof_df["w_phys"].abs())
    oof_z, oof_thr = significance_scan(oof_df["label"].to_numpy(), oof_df["score"].to_numpy(), oof_df["w_phys"].to_numpy())
    print(f"[{label}] OOF weighted AUC (pooled) = {oof_auc:.4f}")
    print(f"[{label}] OOF max Asimov Z (pooled) = {oof_z:.3f} at score cut = {oof_thr:.4f}")

    return oof_df, models, scalers, medians_list


def plot_oof_roc(oof_df, title=""):

    """OOF ROC over the full sample, plus a fold-spread band."""

    y = oof_df["label"].to_numpy()
    s = oof_df["score"].to_numpy()
    w = oof_df["w_phys"].abs().to_numpy()
    fpr, tpr, _ = roc_curve(y, s, sample_weight=w)
    auc = roc_auc_score(y, s, sample_weight=w)

    grid = np.linspace(0, 1, 200)
    fold_tprs = []
    for k in sorted(oof_df["fold"].unique()):
        sub = oof_df[oof_df["fold"] == k]
        if sub["label"].nunique() < 2:
            continue
        f_k, t_k, _ = roc_curve(sub["label"], sub["score"], sample_weight=sub["w_phys"].abs())
        fold_tprs.append(np.interp(grid, f_k, t_k))
    fold_tprs = np.array(fold_tprs)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"OOF ROC (AUC={auc:.4f})")
    if len(fold_tprs):
        ax.fill_between(grid, fold_tprs.min(axis=0), fold_tprs.max(axis=0), color="steelblue",
                         alpha=0.2, label="fold spread (min-max)")
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title=title, xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    plt.show()
    return auc


def save_track_artifacts_gnn(models, scalers, medians_list, oof_df, base_dir, track_name, hyperparams=None):

    """
    GNN analogue of the other NN notebooks' save_track_artifacts_nn.

    NOTE: features/hyperparams/OOF outputs are suffixed with `_gnn` -
    BASE_DIR_RUN2/RUN3/COMB is SHARED with the sibling XGBoost master
    pipeline, which writes its own unsuffixed features_{track}.json/
    hyperparams_{track}.json/oof_scores_{track}.csv to the SAME directory.
    Without the suffix this would silently overwrite (and be overwritten
    by) the XGBoost pipeline's artifacts - the exact collision bug found
    and fixed in DNN_2l2tau.ipynb (see repo memory). model_{track}_fold{k}.pt
    and preprocess_{track}.pkl are left unsuffixed - different extensions
    than XGBoost's own outputs, so no collision there.
    """

    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)
    for k, m in enumerate(models):
        torch.save(m.state_dict(), base_dir / f"model_{track_name}_fold{k}.pt")
    with open(base_dir / f"preprocess_{track_name}.pkl", "wb") as f:
        pickle.dump({"scalers": scalers, "medians": medians_list}, f)
    with open(base_dir / f"features_{track_name}_gnn.json", "w") as f:
        json.dump({"features": REQUIRED_OBJECT_COLUMNS}, f)
    if hyperparams is not None:
        with open(base_dir / f"hyperparams_{track_name}_gnn.json", "w") as f:
            json.dump(hyperparams, f)
    oof_path = base_dir / f"oof_scores_{track_name}_gnn.csv"
    oof_df.to_csv(oof_path, index=False)
    print(f"Saved {len(models)} fold models + preprocessing + feature list + hyperparams + OOF scores "
          f"-> {base_dir}/ (track={track_name})")

## Load Run 2 Data

In [ ]:
candidate_features = discover_common_features(BASE_DIR_RUN2)
data = load_run_data(BASE_DIR_RUN2, candidate_features)
data, features = clean_data(data, candidate_features)
data = assign_folds(data)

print(f"\nLoaded {len(data)} events | {len(features)} leakage-free features available")
print(f"\nFold sizes (fold = eventNumber % {N_FOLDS}):")
print(data["fold"].value_counts().sort_index())

## Sentinel Audit (-1)

In [ ]:
NEG1_SENTINEL_FEATURES = set()

neg1_rows = []
for f in features:
    vals = data[f]
    frac_neg1 = (vals == -1).mean()
    if frac_neg1 == 0:
        continue
    above = vals[vals > -1]
    gap = (above.min() - (-1)) if len(above) else np.nan
    neg1_rows.append({"feature": f, "frac_exactly_-1": frac_neg1, "gap_to_next_value_above": gap})

neg1_df = pd.DataFrame(neg1_rows).sort_values("frac_exactly_-1", ascending=False)
print(f"{len(neg1_df)} / {len(features)} features have at least one row exactly equal to -1:")
print(neg1_df.to_string(index=False))

for f in NEG1_SENTINEL_FEATURES:
    data[f] = data[f].mask(data[f] == -1)

if NEG1_SENTINEL_FEATURES:
    print(f"\nMasked -1 -> NaN for: {sorted(NEG1_SENTINEL_FEATURES)}")
else:
    print("\nNEG1_SENTINEL_FEATURES is empty - no -1 values masked.")

## Object / Node Schema

In [ ]:
OBJECT_COLUMNS = {
    "lepton1": {"pt": "l1_pt", "eta": "l1_eta", "phi": "l1_phi", "e": "l1_e", "charge": "l1_charge", "pdg": "l1_pdg"},
    "lepton2": {"pt": "l2_pt", "eta": "l2_eta", "phi": "l2_phi", "e": "l2_e", "charge": "l2_charge", "pdg": "l2_pdg"},
    "tau1":    {"pt": "tau1_pt", "eta": "tau1_eta", "phi": "tau1_phi"},
    "tau2":    {"pt": "tau2_pt", "eta": "tau2_eta", "phi": "tau2_phi"},
    "jet1":    {"pt": "j1_pt",   "eta": "j1_eta",   "phi": "j1_phi",   "e": "j1_e"},
    "jet2":    {"pt": "j2_pt",   "eta": "j2_eta",   "phi": "j2_phi",   "e": "j2_e"},
    "met":     {"pt": "met_met", "phi": "met_phi",  "e": "met_sumet"},
}
NODE_ORDER = ["lepton1", "lepton2", "tau1", "tau2", "jet1", "jet2", "met"]
NODE_TYPE = {"lepton1": "lepton", "lepton2": "lepton", "tau1": "tau", "tau2": "tau",
             "jet1": "jet", "jet2": "jet", "met": "met"}
TYPE_LIST = ["lepton", "tau", "jet", "met"]
N_NODES = len(NODE_ORDER)
N_NODE_FEATURES = 7 + len(TYPE_LIST)  # pt, eta, sin(phi), cos(phi), e, charge, is_electron + type one-hot

REQUIRED_OBJECT_COLUMNS = sorted({c for cols in OBJECT_COLUMNS.values() for c in cols.values()})
missing = [c for c in REQUIRED_OBJECT_COLUMNS if c not in features]
assert not missing, f"Node schema references columns dropped by clean_data / the leakage policy: {missing}"

# Continuous columns (imputed + standard-scaled per fold, fit on TRAIN only).
# phi is handled separately via sin/cos; charge and pdg-derived is_electron
# are left unscaled (0/1-ish flags - scaling would just relabel them).
CONTINUOUS_NODE_COLS = sorted({
    c for cols in OBJECT_COLUMNS.values() for k, c in cols.items() if k in ("pt", "eta", "e")
})

print(f"{N_NODES} nodes/event x {N_NODE_FEATURES} features/node")
print(f"{len(CONTINUOUS_NODE_COLS)} continuous columns to impute + scale: {CONTINUOUS_NODE_COLS}")

# Fully-connected edge set - identical topology for every event (n_jet >= 2
# in the preselection guarantees all 6 slots are always populated), so it's
# built once and shared across every graph/fold/run.
EDGE_INDEX = torch.tensor(
    [[i, j] for i in range(N_NODES) for j in range(N_NODES) if i != j],
    dtype=torch.long,
).t().contiguous().to(DEVICE)

# ---- Batched edge_index / batch-vector helpers -----------------------------
# Every event shares the EXACT SAME graph topology, so a whole batch's
# disjoint-union edge_index and node->graph `batch` vector can be built with
# plain vectorized tensor ops (broadcasting/repeat), never a Python loop over
# individual graphs - this is what lets training skip PyG's DataLoader
# collation entirely (see the dataset-building / run_epoch cells below).
# Cached per batch size since the same size is reused every batch/epoch.
_edge_index_cache = {}
_batch_vector_cache = {}


def get_batched_edge_index(batch_size):

    """Disjoint-union `edge_index` for `batch_size` copies of EDGE_INDEX,
    node indices offset by i * N_NODES per graph i."""

    if batch_size not in _edge_index_cache:
        offsets = torch.arange(batch_size, device=DEVICE).repeat_interleave(EDGE_INDEX.shape[1]) * N_NODES
        _edge_index_cache[batch_size] = EDGE_INDEX.repeat(1, batch_size) + offsets
    return _edge_index_cache[batch_size]


def get_batch_vector(batch_size):

    """`batch` vector (which graph each of the batch_size * N_NODES rows of
    a flattened node tensor belongs to), matching PyG's `Batch.batch` convention."""

    if batch_size not in _batch_vector_cache:
        _batch_vector_cache[batch_size] = torch.arange(batch_size, device=DEVICE).repeat_interleave(N_NODES)
    return _batch_vector_cache[batch_size]

## Graph Construction

Defines `stack_node_features`/`build_graph_tensors` - pure, data-independent
functions reused unchanged by every fold, and by the Run 3/Combined sections
below. Actual tensor construction now happens per-fold, inside
`prepare_fold_tensors_gnn` (K-Fold helper library above), instead of once
against a single fixed train/val/test split.

In [ ]:
def stack_node_features(df_scaled, df_imp):

    """
    Vectorized construction of the (n_events, N_NODES, N_NODE_FEATURES) node
    tensor. `df_scaled` supplies pt/eta/e (imputed + standard-scaled);
    `df_imp` supplies phi/charge/pdg (imputed but NOT scaled - phi goes
    through sin/cos, charge/pdg are categorical-ish and left raw).
    """

    n = len(df_scaled)
    node_arrays = []

    for name in NODE_ORDER:
        cols = OBJECT_COLUMNS[name]

        pt = df_scaled[cols["pt"]].to_numpy(dtype=np.float32)
        eta = df_scaled[cols["eta"]].to_numpy(dtype=np.float32) if "eta" in cols else np.zeros(n, dtype=np.float32)
        phi = df_imp[cols["phi"]].to_numpy(dtype=np.float32)
        e = df_scaled[cols["e"]].to_numpy(dtype=np.float32) if "e" in cols else np.zeros(n, dtype=np.float32)
        charge = df_imp[cols["charge"]].to_numpy(dtype=np.float32) if "charge" in cols else np.zeros(n, dtype=np.float32)
        is_electron = (
            (np.abs(df_imp[cols["pdg"]].to_numpy()) == 11).astype(np.float32)
            if "pdg" in cols else np.zeros(n, dtype=np.float32)
        )
        type_onehot = np.tile(
            np.array([float(NODE_TYPE[name] == t) for t in TYPE_LIST], dtype=np.float32), (n, 1)
        )

        node_arrays.append(np.column_stack(
            [pt, eta, np.sin(phi), np.cos(phi), e, charge, is_electron, type_onehot]
        ))

    return np.stack(node_arrays, axis=1)  # (n, N_NODES, N_NODE_FEATURES)


def build_graph_tensors(df_scaled, df_imp, df_raw, fit_weights=None):

    """
    Vectorized construction of GPU-resident tensors for one split: `x`
    (n_events, N_NODES, N_NODE_FEATURES), `y` (n_events,), `w_fit` (training
    weights, or None), and `w_abs` (plain |w_phys|, always). Every event
    shares the EXACT SAME graph topology (`EDGE_INDEX`/`N_NODES` are
    fixed), so there's no need for PyG's dynamic-graph collation machinery
    at all. If `fit_weights` is given (class-balanced + mean-normalized
    training weights), `w_fit` holds those (paired with `w_abs` for
    eval-mode metrics on the same split); otherwise `w_fit` is None and
    only `w_abs` is meaningful (the val/test convention).
    """

    x = torch.from_numpy(stack_node_features(df_scaled, df_imp)).to(DEVICE)
    y = torch.from_numpy(df_raw["label"].to_numpy(dtype=np.float32)).to(DEVICE)
    w_abs = torch.from_numpy(np.abs(df_raw["w_phys"].to_numpy(dtype=np.float32))).to(DEVICE)
    w_fit = torch.from_numpy(fit_weights.astype(np.float32)).to(DEVICE) if fit_weights is not None else None

    return x, y, w_fit, w_abs

## Fold-0 Preview: Preprocessing & Weight-Balance Diagnostics

Same convention as `DNN.ipynb`: fold 0 (`test=fold 0, val=fold 1, train=the
other 3 folds`) stands in as a REPRESENTATIVE preview for the sections below
(GNN model/training-loop smoke test, permutation importance, the
significance-scan cut). The actual production result is the 5-fold
out-of-fold (OOF) run in the "K-Fold Production Run & Artifacts" section
near the end of this run's section, which scores every event with a model
that never trained on it.

In [ ]:
PLOTS_DIR_R2 = BASE_DIR_RUN2 / "plots"
PLOTS_DIR_R2.mkdir(parents=True, exist_ok=True)

target_yields = compute_process_yield_targets(data)

fd0 = prepare_fold_tensors_gnn(data, target_yields, cell_cols=(), n_folds=N_FOLDS, k=0)

w_before = fd0["train_df"]["w_phys"].to_numpy()
w_after = fd0["w_train_fit"]
y_preview = fd0["train_df"]["label"].to_numpy()

plot_weight_balance(
    y_preview, w_before, w_after,
    title="Run 2 (GNN) - make_fit_weights balancing (fold 0 train)",
    save_path=PLOTS_DIR_R2 / "Run2WeightBalance_GNN.png",
)

print(f"Built {fd0['X_train_t'].shape[0]} train / {fd0['X_val_t'].shape[0]} val / "
      f"{fd0['X_test_t'].shape[0]} test graphs (fold 0 preview, GPU-resident tensors)")
print(f"Node tensor shape per split: (n_events, {N_NODES}, {N_NODE_FEATURES})")

print("\nN_eff (training sample, positive-only, post yield-rescale) by label - fold 0:")
print(n_eff_table(fd0["train_df"], ["label"]))
print("\nN_eff (eval sample, signed, FULL fold-0 test partition) by label:")
print(n_eff_table(fd0["test_df"], ["label"]))
print(f"\nDropped {fd0['n_dropped_train']} negative-w_phys training rows in this "
      f"preview fold (kept, not abs'd, elsewhere - see prepare_fold_data).")

## GNN Model

In [ ]:
BATCH_SIZE = 8192
DEFAULT_HIDDEN_CHANNELS = 64
DEFAULT_DROPOUT = 0.3


class ObjectGNN(nn.Module):

    """
    Object-level graph classifier: 2x GATv2Conv, mean+max pooling, MLP head,
    single output logit (paired with BCEWithLogitsLoss for numerical
    stability, same convention as DNN.ipynb's SimpleMLP).
    """

    def __init__(self, in_channels, hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT):

        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels)
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.head(x).squeeze(-1)


def build_model(hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT, in_channels=None):

    """Factory so training/hyperparameter code can build a fresh model per trial."""

    if in_channels is None:
        in_channels = N_NODE_FEATURES
    return ObjectGNN(in_channels=in_channels, hidden_channels=hidden_channels, dropout=dropout).to(DEVICE)


model = build_model()
print(model)

## Training Loop

In [ ]:
N_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 1e-3


def run_epoch(model, X, y, w, criterion, optimizer, train, batch_size=BATCH_SIZE):

    """
    One pass over (X, y, w) in `batch_size`-sized chunks, optionally taking
    a gradient step. Batches via GPU-side tensor indexing (`torch.randperm`
    for shuffling on the train pass) instead of a PyG `DataLoader`, and the
    batched `edge_index`/`batch` vector come from the cached
    `get_batched_edge_index`/`get_batch_vector` helpers (vectorized, no
    per-graph Python loop). Only syncs to CPU ONCE at the end of the whole
    epoch (`.cpu().numpy()` below), not per batch - avoids both the CPU-bound
    per-batch graph collation AND the CUDA-sync overhead of a per-batch
    DataLoader-based loop (each `.item()`/`.cpu()` call forces a full sync).
    """

    model.train(train)
    n = X.shape[0]
    order = torch.randperm(n, device=DEVICE) if train else torch.arange(n, device=DEVICE)

    total_loss = torch.zeros((), device=DEVICE)
    total_weight = torch.zeros((), device=DEVICE)
    all_labels, all_probs, all_weights = [], [], []

    with torch.set_grad_enabled(train):
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            b = idx.shape[0]

            bx = X[idx].reshape(b * N_NODES, N_NODE_FEATURES)
            by = y[idx]
            bw = w[idx]

            logits = model(bx, get_batched_edge_index(b), get_batch_vector(b))
            loss = (criterion(logits, by) * bw).sum() / bw.sum()

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bw_sum = bw.sum().detach()
            total_loss += loss.detach() * bw_sum
            total_weight += bw_sum
            all_labels.append(by.detach())
            all_probs.append(torch.sigmoid(logits).detach())
            all_weights.append(bw.detach())

    labels = torch.cat(all_labels).cpu().numpy()
    probs = torch.cat(all_probs).cpu().numpy()
    weights = torch.cat(all_weights).cpu().numpy()
    auc = roc_auc_score(labels, probs, sample_weight=weights)

    return (total_loss / total_weight).item(), auc


def score_dataset(model, X, batch_size=BATCH_SIZE):

    """
    Batched inference over `X` (n_events, N_NODES, N_NODE_FEATURES),
    returning sigmoid probabilities as a 1-D GPU tensor. Reused by every
    val/test evaluation cell below, and by run_kfold_gnn, instead of a PyG
    DataLoader loop.
    """

    model.eval()
    all_p = []
    with torch.no_grad():
        for start in range(0, X.shape[0], batch_size):
            bx = X[start:start + batch_size]
            b = bx.shape[0]
            logits = model(bx.reshape(b * N_NODES, N_NODE_FEATURES), get_batched_edge_index(b), get_batch_vector(b))
            all_p.append(torch.sigmoid(logits))
    return torch.cat(all_p)


def train_model(X_train_data, y_train_data, w_train_fit_data, w_train_eval_data,
                 X_val_data, y_val_data, w_val_data,
                 hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT, lr=LEARNING_RATE,
                 batch_size=BATCH_SIZE, n_epochs=N_EPOCHS, patience=PATIENCE, verbose=True):

    """
    Build a fresh ObjectGNN and train it with early stopping on weighted
    validation AUC. All data args are REQUIRED (no more defaulting to fixed
    module-level globals) since every fold has its own tensors. `batch_size`
    is a tunable hyperparameter (Optuna search below) - defaults to the
    global BATCH_SIZE for any caller that doesn't tune it. Returns
    (model, history, best_val_auc, best_train_auc, best_train_auc_eval) -
    `model` already has the best-epoch weights loaded, and the two
    train_auc numbers are read off that SAME epoch.
    """

    trial_model = build_model(hidden_channels=hidden_channels, dropout=dropout)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.Adam(trial_model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": [],
               "train_loss_eval": [], "train_auc_eval": []}
    best_val_auc, best_train_auc, best_train_auc_eval = -np.inf, None, None
    best_state, epochs_no_improve = None, 0

    for epoch in range(1, n_epochs + 1):

        train_loss, train_auc = run_epoch(trial_model, X_train_data, y_train_data, w_train_fit_data, criterion, optimizer, train=True, batch_size=batch_size)
        train_loss_eval, train_auc_eval = run_epoch(trial_model, X_train_data, y_train_data, w_train_eval_data, criterion, optimizer, train=False, batch_size=batch_size)
        val_loss, val_auc = run_epoch(trial_model, X_val_data, y_val_data, w_val_data, criterion, optimizer, train=False, batch_size=batch_size)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)
        history["train_loss_eval"].append(train_loss_eval)
        history["train_auc_eval"].append(train_auc_eval)

        if verbose:
            print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"| train_auc={train_auc:.4f} train_auc_eval={train_auc_eval:.4f} val_auc={val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc, best_train_auc, best_train_auc_eval = val_auc, train_auc, train_auc_eval
            best_state, epochs_no_improve = {k: v.clone() for k, v in trial_model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (best val_auc={best_val_auc:.4f})")
                break

    trial_model.load_state_dict(best_state)
    return trial_model, history, best_val_auc, best_train_auc, best_train_auc_eval


# ---- Fold-0 preview baseline (default hyperparameters) - used only to
# report the numbers in the sections below; the actual production model is
# the 5-fold OOF ensemble built in the K-Fold Production Run section further
# down. Refit with Optuna-tuned hyperparameters in the "Optuna Hyperparameter
# Search" section right after this one.
model, history, best_val_auc, best_train_auc, best_train_auc_eval = train_model(
    fd0["X_train_t"], fd0["y_train_t"], fd0["w_train_fit_t"], fd0["w_train_abs_t"],
    fd0["X_val_t"], fd0["y_val_t"], fd0["w_val_abs_t"],
)

print(f"\nBest val_auc = {best_val_auc:.4f} | train_auc (dropout on) = {best_train_auc:.4f} "
      f"| train_auc_eval (dropout off, comparable) = {best_train_auc_eval:.4f}")

## Optuna Hyperparameter Search

Bayesian search (TPE sampler + median pruning) over `hidden_channels`,
`dropout`, `lr`, `batch_size` (categorical, starting at 128) and `patience`,
scored by mean expected significance (Asimov Z, high-score tail,
`significance_scan`) - the SAME objective family used by the XGBoost master
pipelines and the MLP/DNN/PNN notebooks, so the model comparison is tuned
consistently (see `run_optuna_search_gnn`'s docstring for the accepted-CV-
leak caveat, which mirrors XGBoost's `run_optuna_search`). Runs on
`N_OPTUNA_INNER_FOLDS` folds (fewer than the final `N_FOLDS` rotation) since
GNN training is much more expensive per-fold than XGBoost's;
`trial.report`/pruning happens per-EPOCH (not per-fold) for the same
reason. Hyperparameters are frozen once here and reused for all `N_FOLDS`
outer folds later (`run_kfold_gnn`) - never re-tuned per fold. Unlike the
MLP/DNN notebooks, `weight_decay` is NOT tuned here (kept at Adam's default
of 0, matching this notebook's original untuned `train_model`).

In [ ]:
def train_model_for_search(trial, step_offset, hidden_channels, dropout, lr, batch_size, patience,
                            X_train_data, y_train_data, w_train_data,
                            X_val_data, y_val_data, w_val_data, w_val_signed, n_epochs=40):

    """
    Optuna-search-only training loop: same architecture/early-stopping
    convention as `train_model` (best-val_auc checkpoint restored, patience
    on val_auc), but additionally reports the per-EPOCH validation
    significance (Asimov Z, via `significance_scan` on SIGNED w_phys) to
    `trial` and raises `optuna.TrialPruned()` if the pruner says to stop -
    GNN training is much more expensive than XGBoost's, so per-epoch (not
    per-fold) reporting is what makes pruning actually save time here.
    `step_offset` lets the caller give every fold's epochs a distinct,
    monotonically increasing `step` within the same trial (Optuna's
    report/prune bookkeeping is keyed by trial+step, so two folds both
    starting at step=1 would collide). Early stopping itself still tracks
    val AUC (matching `train_model`'s convention) - only the
    OPTUNA-VISIBLE metric is significance. Returns the best-epoch
    significance Z reached in this fold and the number of epochs run.
    """

    trial_model = build_model(hidden_channels=hidden_channels, dropout=dropout)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.Adam(trial_model.parameters(), lr=lr)

    best_val_auc, best_z, epochs_no_improve = -np.inf, 0.0, 0

    for epoch in range(1, n_epochs + 1):
        run_epoch(trial_model, X_train_data, y_train_data, w_train_data, criterion, optimizer, train=True, batch_size=batch_size)
        _, val_auc = run_epoch(trial_model, X_val_data, y_val_data, w_val_data, criterion, optimizer, train=False, batch_size=batch_size)

        val_scores = score_dataset(trial_model, X_val_data, batch_size=batch_size).cpu().numpy()
        z, _ = significance_scan(y_val_data.cpu().numpy(), val_scores, w_val_signed)

        trial.report(z, step=step_offset + epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_auc > best_val_auc:
            best_val_auc, best_z, epochs_no_improve = val_auc, z, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    return best_z, epoch


def run_optuna_search_gnn(data, target_yields, cell_cols=(),
                           n_trials=N_OPTUNA_TRIALS, n_inner_folds=N_OPTUNA_INNER_FOLDS,
                           n_epochs=40, study_name="gnn_opt", seed=RANDOM_STATE):

    """
    Bayesian hyperparameter search over hidden_channels/dropout/lr/
    batch_size/patience, scored by mean per-fold validation significance
    (Asimov Z) - SAME objective family as XGBoost's `run_optuna_search` and
    the MLP/DNN notebooks' `run_optuna_search_nn`, for a consistent
    cross-model comparison.

    KNOWN, ACCEPTED LEAK - NOT NESTED CV (same caveat as the XGBoost
    pipelines' run_optuna_search): hyperparameters are chosen using
    `n_inner_folds` folds' mean Z, and (a subset of) those SAME folds later
    contribute to the final OOF scores via `run_kfold_gnn`. Deliberately
    accepted as the standard practical compromise. `n_inner_folds` is
    smaller than the final `N_FOLDS` rotation (an efficiency tradeoff, GNN
    training being far more expensive per-fold than XGBoost's) - the final
    `run_kfold_gnn` rotation still uses the full `N_FOLDS`.
    """

    def objective(trial):
        hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512, 1024, 2048, 4096, 8192, 16384])
        patience = trial.suggest_int("patience", 5, 15)

        fold_zs, step_offset = [], 0
        for k in range(n_inner_folds):
            fd = prepare_fold_tensors_gnn(data, target_yields, cell_cols=cell_cols, n_folds=N_FOLDS, k=k)
            w_val_signed = fd["val_df"]["w_phys"].to_numpy()
            z, epochs_run = train_model_for_search(
                trial, step_offset, hidden_channels, dropout, lr, batch_size, patience,
                fd["X_train_t"], fd["y_train_t"], fd["w_train_fit_t"],
                fd["X_val_t"], fd["y_val_t"], fd["w_val_abs_t"], w_val_signed, n_epochs=n_epochs,
            )
            fold_zs.append(z)
            step_offset += epochs_run

        return float(np.mean(fold_zs))

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=study_name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, gc_after_trial=True)

    print(f"\n[{study_name}] completed trials: {len(study.trials)}")
    print(f"[{study_name}] best mean CV significance Z ({n_inner_folds} eventNumber-folds): {study.best_value:.5f}")
    print(f"[{study_name}] best parameters:")
    for name, value in study.best_params.items():
        print(f"    {name}: {value}")
    return study


def params_from_study_gnn(study):

    """Extract the frozen hyperparameter dict from a completed Optuna study,
    ready to pass into `run_kfold_gnn`. Hyperparameters are frozen across
    all outer folds - never re-tuned per fold."""

    return dict(study.best_params)


def plot_optuna_diagnostics_gnn(study, title_suffix=""):

    """2-panel diagnostic: Optuna trial history + hyperparameter importance
    (same convention as the XGBoost/MLP/DNN pipelines' plot_optuna_diagnostics)."""

    trials = study.trials_dataframe()
    complete = trials.loc[trials["state"] == "COMPLETE"].copy()
    param_importance = optuna.importance.get_param_importances(study)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    axes[0].plot(complete["number"], complete["value"], "o", alpha=0.7)
    axes[0].plot(complete["number"], complete["value"].cummax(), color="black", linewidth=2, label="best so far")
    axes[0].set(xlabel="Trial", ylabel="Mean CV significance Z (inner eventNumber-folds)", title=f"Optuna history {title_suffix}")
    axes[0].legend()

    names = list(param_importance)
    values = [param_importance[name] for name in names]
    axes[1].barh(names[::-1], values[::-1], color="steelblue")
    axes[1].set(xlabel="Optuna parameter importance", title="Hyperparameter importance")

    plt.show()


study_run2 = run_optuna_search_gnn(data, target_yields, study_name="gnn_opt_run2")
best_params_run2 = params_from_study_gnn(study_run2)
plot_optuna_diagnostics_gnn(study_run2, title_suffix="(2l2tau, Run 2, GNN)")

# ---- Refit the fold-0 preview model with the tuned hyperparameters, so the
# rest of this track's (feature-importance/physics-FOM) cells below reflect
# the tuned architecture, not the arbitrary defaults.
model, history, best_val_auc, best_train_auc, best_train_auc_eval = train_model(
    fd0["X_train_t"], fd0["y_train_t"], fd0["w_train_fit_t"], fd0["w_train_abs_t"],
    fd0["X_val_t"], fd0["y_val_t"], fd0["w_val_abs_t"],
    hidden_channels=best_params_run2["hidden_channels"], dropout=best_params_run2["dropout"],
    lr=best_params_run2["lr"], batch_size=best_params_run2["batch_size"],
    patience=best_params_run2["patience"], verbose=False,
)
print(f"\nTuned fold-0 preview: val_auc={best_val_auc:.4f} | hidden_channels={best_params_run2['hidden_channels']} "
      f"| batch_size={best_params_run2['batch_size']}")

## Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[0].plot(history["train_loss_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[0].plot(history["val_loss"], label="val", color="tab:orange")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("weighted BCE loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["train_auc"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[1].plot(history["train_auc_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[1].plot(history["val_auc"], label="val", color="tab:orange")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("weighted AUC")
axes[1].set_title("AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

y_val_scores = score_dataset(model, fd0["X_val_t"]).cpu().numpy()
y_val = fd0["y_val_t"].cpu().numpy()
w_val = fd0["w_val_abs_t"].cpu().numpy()

auc_val = roc_auc_score(y_val, y_val_scores, sample_weight=w_val)
print(f"Final weighted val AUC (fold-0 preview) = {auc_val:.4f}")

fpr, tpr, _ = roc_curve(y_val, y_val_scores, sample_weight=w_val)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"GNN (AUC = {auc_val:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve (validation, fold-0 preview)")
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance (Permutation)

XGBoost has a built-in gain-based importance, but a neural net doesn't - the
model-agnostic equivalent is **permutation importance**: shuffle one
feature's column across events (destroying its relationship with the
label) and measure how much the weighted validation AUC drops. Repeated a
few times per feature and averaged for stability. Computed on the
validation set so it isn't inflated by the model simply memorizing train.
Applied here to the raw per-object columns that feed the node-feature
tensor (~23 of these, not ~80 flat features).

In [ ]:
def score_node_features(model, node_features_np, chunk_size=4096):

    """
    Batched forward pass over a (n, N_NODES, N_NODE_FEATURES) node array.
    All graphs share the exact same EDGE_INDEX topology (the preselection
    guarantees all 6 node slots are always populated), so instead of
    constructing one torch_geometric Data object per event (slow in Python
    for tens of thousands of events, repeated ~100+ times below), each
    chunk is manually batched as a single big graph by offsetting node
    indices - verified to give bit-identical output to PyG's own batching.
    """

    model.eval()
    n = node_features_np.shape[0]
    probs = np.empty(n, dtype=np.float32)
    edge_index_cpu = EDGE_INDEX

    for start in range(0, n, chunk_size):
        chunk = node_features_np[start:start + chunk_size]
        b = chunk.shape[0]
        x = torch.from_numpy(chunk.reshape(-1, N_NODE_FEATURES)).to(DEVICE)
        offsets = torch.arange(b, device=DEVICE) * N_NODES
        edge_index_chunk = (edge_index_cpu[:, None, :] + offsets[None, :, None]).reshape(2, -1).to(DEVICE)
        batch_vec = torch.repeat_interleave(torch.arange(b, device=DEVICE), N_NODES)
        with torch.no_grad():
            logits = model(x, edge_index_chunk, batch_vec)
        probs[start:start + b] = torch.sigmoid(logits).cpu().numpy()

    return probs


def permutation_importance_gnn(model, val_scaled, val_imp, y, w, columns, n_repeats=5, random_state=RANDOM_STATE):

    """
    GNN analogue of DNN.ipynb's permutation_importance, applied to the raw
    per-object columns that feed the node-feature tensor. For each column,
    shuffle it across validation events `n_repeats` times and measure the
    average drop in weighted AUC relative to the unshuffled baseline.
    Returns a pandas Series (column -> mean AUC drop), sorted descending.
    """

    rng = np.random.default_rng(random_state)
    n = len(val_scaled)

    baseline_probs = score_node_features(model, stack_node_features(val_scaled, val_imp))
    baseline_auc = roc_auc_score(y, baseline_probs, sample_weight=w)
    print(f"Baseline weighted val AUC: {baseline_auc:.4f}")

    mean_drops = []
    for col in columns:
        drops = []
        for _ in range(n_repeats):
            perm = rng.permutation(n)
            vs, vi = val_scaled.copy(), val_imp.copy()
            if col in vs.columns:
                vs[col] = vs[col].to_numpy()[perm]
            if col in vi.columns:
                vi[col] = vi[col].to_numpy()[perm]
            probs = score_node_features(model, stack_node_features(vs, vi))
            drops.append(baseline_auc - roc_auc_score(y, probs, sample_weight=w))
        mean_drops.append(np.mean(drops))

    return pd.Series(mean_drops, index=columns, name="auc_drop").sort_values(ascending=False)


def plot_importance_bar(imp, top_n=30, title="", color="lightblue", save_path=None):

    """
    Horizontal bar chart of the top `top_n` columns by importance (same
    convention as DNN.ipynb's plot_importance_bar). If `save_path` is
    given, the figure is written to disk (parent directories created as
    needed, dpi=150) before being displayed.
    """

    n = min(top_n, len(imp))

    fig, ax = plt.subplots(figsize=(8, max(4, 0.28 * n)))
    imp.head(top_n)[::-1].plot.barh(ax=ax, color=color)
    ax.set_xlabel("Weighted AUC drop when shuffled")
    ax.set_title(title)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        print(f"Saved plot -> {save_path}")

    plt.show()

In [ ]:
importance = permutation_importance_gnn(
    model, fd0["val_scaled"], fd0["val_imp"], y_val, w_val, REQUIRED_OBJECT_COLUMNS, n_repeats=5,
)
plot_importance_bar(
    importance, top_n=30,
    title=f"GNN permutation importance — {len(REQUIRED_OBJECT_COLUMNS)} per-object input columns (2l2tau, Run 2, fold-0 preview)",
)
importance

## Physics Figure of Merit

Weighted AUC is a global ranking metric; for HH what matters is
significance in the high-score region. `significance_scan` (defined in the
K-Fold helper library above) reports the max-Asimov-significance score cut
on this fold-0 preview's VAL split - the same convention `DNN.ipynb` uses.
There is no single held-out test set to score here anymore: that role is
now filled by the 5-fold out-of-fold (OOF) production run below, which
scores every event with a model that never trained on it and applies this
VAL-selected cut (`thr_val`) frozen, not re-scanned.

In [ ]:
w_val_signed = fd0["val_df"]["w_phys"].to_numpy()
z_val, thr_val = significance_scan(y_val, y_val_scores, w_val_signed)

print(f"Weighted val AUC (fold-0 preview) = {auc_val:.4f}")
print(f"Max Asimov Z (val, fold-0 preview) = {z_val:.3f} at score cut = {thr_val:.4f}")

### K-Fold Production Run & Artifacts (Run 2)

Everything above (fold-0 preview training, permutation importance, the
significance scan) used only fold 0 as a representative preview - the same
convention `DNN.ipynb` uses. This is where the actual 5-fold out-of-fold
(OOF) production run happens: `run_kfold_gnn` trains one `ObjectGNN` per
outer fold (frozen default hyperparameters - this notebook has no Optuna
search) and scores every event with a model that never trained on it.
Replaces the old single 80/10/10 split's "Held-Out Test Evaluation" cell -
there is no single held-out test set anymore, only the pooled OOF metrics
below, evaluated at the fold-0-preview VAL-selected score cut (`thr_val`,
frozen, not re-scanned).

In [ ]:
# ---- K-FOLD PRODUCTION RUN (Run 2) -----------------------------------------
oof_df, models, scalers, medians = run_kfold_gnn(
    data, target_yields, cell_cols=(), n_folds=N_FOLDS, label="Run2",
    hidden_channels=best_params_run2["hidden_channels"], dropout=best_params_run2["dropout"],
    lr=best_params_run2["lr"], batch_size=best_params_run2["batch_size"],
    patience=best_params_run2["patience"],
)

oof_auc = plot_oof_roc(oof_df, title="2l2tau Run 2 (GNN) - OOF ROC")

save_track_artifacts_gnn(models, scalers, medians, oof_df, BASE_DIR_RUN2, "run2", hyperparams=best_params_run2)

# ---- Apply the fold-0-preview VAL-selected score cut (thr_val) to the OOF
# scores, FROZEN (not re-scanned) - every OOF score comes from a fold model
# that never trained on that row, and the cut itself was chosen only from
# the fold-0 preview's VAL partition.
sel_test = oof_df["score"] >= thr_val
S_test = oof_df.loc[sel_test & (oof_df["label"] == 1), "w_phys"].sum()
B_test = oof_df.loc[sel_test & (oof_df["label"] == 0), "w_phys"].sum()
z_test = (np.sqrt(2 * ((S_test + B_test) * np.log(1 + S_test / B_test) - S_test))
          if (S_test > 0 and B_test > 0) else np.nan)

print(f"\nWeighted AUC (Run 2): val (fold-0 preview) = {auc_val:.4f}  |  "
      f"OOF (5-fold, scored once per event) = {oof_auc:.4f}")
print(f"At the fold-0-preview VAL-selected score cut = {thr_val:.4f} (frozen, NOT re-scanned on OOF):")
print(f"  OOF S = {S_test:.2f} | OOF B = {B_test:.2f} | OOF Z = {z_test:.3f}  (val Z was {z_val:.3f})")

## Sanity Checks & Summary

In [ ]:
# ---- Determinism check (same acceptance criterion as DNN.ipynb): re-seeding
# immediately before each of two short training runs should give an EXACT
# match, proving set_seed()/use_deterministic_algorithms(True) pin every
# source of randomness. Cheap config (small model, 5 epochs) purely to keep
# this check fast - not meant to reflect final model quality.

set_seed(RANDOM_STATE)
_, _, det_check_a, _, _ = train_model(
    fd0["X_train_t"], fd0["y_train_t"], fd0["w_train_fit_t"], fd0["w_train_abs_t"],
    fd0["X_val_t"], fd0["y_val_t"], fd0["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)

set_seed(RANDOM_STATE)
_, _, det_check_b, _, _ = train_model(
    fd0["X_train_t"], fd0["y_train_t"], fd0["w_train_fit_t"], fd0["w_train_abs_t"],
    fd0["X_val_t"], fd0["y_val_t"], fd0["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)
assert det_check_a == det_check_b, f"Determinism check FAILED: {det_check_a} != {det_check_b}"

print(f"Determinism check passed: two re-seeded runs give identical val_auc = {det_check_a:.6f}")

# ---- Final summary ----------------------------------------------------------

summary = pd.DataFrame([{
    "model": f"GNN (object nodes, {N_NODES} nodes/event)",
    "val_auc (fold-0 preview)": auc_val,
    "max_asimov_Z (val, fold-0 preview)": z_val,
    "oof_auc (5-fold, scored once per event)": oof_auc,
    "oof_Z_at_val_cut": z_test,
}])

print("\nFinal summary (GNN, 2l2tau Run 2):")
print(summary.to_string(index=False))
summary

## Run 3 (Separate Downstream Section)

Run 2 remains the first complete pass above. This section switches the
active dataset to Run 3 and reuses every function/class defined in the Run 2
section above unchanged (`discover_common_features`, `load_run_data`,
`clean_data`, `assign_folds`, `compute_process_yield_targets`,
`prepare_fold_tensors_gnn`, `run_kfold_gnn`, `plot_oof_roc`,
`save_track_artifacts_gnn`, the `OBJECT_COLUMNS`/`NODE_ORDER`/`EDGE_INDEX`
node schema, `stack_node_features`, `build_graph_tensors`, `ObjectGNN`/
`build_model`, `train_model`/`run_epoch`/`score_dataset`,
`permutation_importance_gnn`/`score_node_features`/`plot_importance_bar`,
`significance_scan`) - only the data and everything derived from it get
`_run3`-suffixed variable names, so the Run 2 results above stay untouched.

## Load Run 3 Data

In [ ]:
candidate_features_run3 = discover_common_features(BASE_DIR_RUN3)
data_run3 = load_run_data(BASE_DIR_RUN3, candidate_features_run3)
data_run3, features_run3 = clean_data(data_run3, candidate_features_run3)
data_run3 = assign_folds(data_run3)

BASE_DIR = BASE_DIR_RUN3
ACTIVE_RUN = "Run 3"

print(f"\nLoaded {len(data_run3)} events | {len(features_run3)} leakage-free features available (Run 3)")
print(f"\nFold sizes (fold = eventNumber % {N_FOLDS}), Run 3:")
print(data_run3["fold"].value_counts().sort_index())

## Sentinel Audit (-1) — Run 3

Same diagnostic as the Run 2 section above, run separately on `data_run3`/
`features_run3`.

In [ ]:
NEG1_SENTINEL_FEATURES_run3 = set()

neg1_rows_run3 = []
for f in features_run3:
    vals = data_run3[f]
    frac_neg1 = (vals == -1).mean()
    if frac_neg1 == 0:
        continue
    above = vals[vals > -1]
    gap = (above.min() - (-1)) if len(above) else np.nan
    neg1_rows_run3.append({"feature": f, "frac_exactly_-1": frac_neg1, "gap_to_next_value_above": gap})

neg1_df_run3 = pd.DataFrame(neg1_rows_run3).sort_values("frac_exactly_-1", ascending=False)
print(f"{len(neg1_df_run3)} / {len(features_run3)} features have at least one row exactly equal to -1:")
print(neg1_df_run3.to_string(index=False))

for f in NEG1_SENTINEL_FEATURES_run3:
    data_run3[f] = data_run3[f].mask(data_run3[f] == -1)

if NEG1_SENTINEL_FEATURES_run3:
    print(f"\nMasked -1 -> NaN for: {sorted(NEG1_SENTINEL_FEATURES_run3)}")
else:
    print("\nNEG1_SENTINEL_FEATURES_run3 is empty - no -1 values masked.")

## Fold-0 Preview: Preprocessing & Weight-Balance Diagnostics — Run 3

The object/node schema (`OBJECT_COLUMNS`, `NODE_ORDER`, `NODE_TYPE`,
`TYPE_LIST`, `N_NODES`, `N_NODE_FEATURES`, `EDGE_INDEX`,
`CONTINUOUS_NODE_COLS`) is run-independent - defined once in the Run 2
section and reused as-is. `prepare_fold_tensors_gnn` refits the median
imputer + `StandardScaler` on Run 3's own fold-0 train split, same
convention as Run 2.

In [ ]:
missing_run3 = [c for c in REQUIRED_OBJECT_COLUMNS if c not in features_run3]
assert not missing_run3, f"Node schema references columns dropped by clean_data / the leakage policy (Run 3): {missing_run3}"

PLOTS_DIR_R3 = BASE_DIR_RUN3 / "plots"
PLOTS_DIR_R3.mkdir(parents=True, exist_ok=True)

target_yields_run3 = compute_process_yield_targets(data_run3)

fd0_run3 = prepare_fold_tensors_gnn(data_run3, target_yields_run3, cell_cols=(), n_folds=N_FOLDS, k=0)

w_before_run3 = fd0_run3["train_df"]["w_phys"].to_numpy()
w_after_run3 = fd0_run3["w_train_fit"]
y_preview_run3 = fd0_run3["train_df"]["label"].to_numpy()

plot_weight_balance(
    y_preview_run3, w_before_run3, w_after_run3,
    title="Run 3 (GNN) - make_fit_weights balancing (fold 0 train)",
    save_path=PLOTS_DIR_R3 / "Run3WeightBalance_GNN.png",
)

print(f"Built {fd0_run3['X_train_t'].shape[0]} train / {fd0_run3['X_val_t'].shape[0]} val / "
      f"{fd0_run3['X_test_t'].shape[0]} test graphs (Run 3, fold 0 preview)")

print("\nN_eff (training sample, positive-only, post yield-rescale) by label - fold 0:")
print(n_eff_table(fd0_run3["train_df"], ["label"]))
print("\nN_eff (eval sample, signed, FULL fold-0 test partition) by label:")
print(n_eff_table(fd0_run3["test_df"], ["label"]))
print(f"\nDropped {fd0_run3['n_dropped_train']} negative-w_phys training rows in this "
      f"preview fold (kept, not abs'd, elsewhere - see prepare_fold_data).")

## Training Loop — Run 3

Reuses `train_model`/`run_epoch`/`build_model`/`ObjectGNN` from the Run 2
section unchanged, with explicit `fd0_run3` tensors passed in so the Run 2
model/history are untouched.

In [ ]:
model_run3, history_run3, best_val_auc_run3, best_train_auc_run3, best_train_auc_eval_run3 = train_model(
    fd0_run3["X_train_t"], fd0_run3["y_train_t"], fd0_run3["w_train_fit_t"], fd0_run3["w_train_abs_t"],
    fd0_run3["X_val_t"], fd0_run3["y_val_t"], fd0_run3["w_val_abs_t"],
)

print(f"\nBest val_auc = {best_val_auc_run3:.4f} | train_auc (dropout on) = {best_train_auc_run3:.4f} "
      f"| train_auc_eval (dropout off, comparable) = {best_train_auc_eval_run3:.4f}")

## Optuna Hyperparameter Search — Run 3

Reuses `train_model_for_search`/`run_optuna_search_gnn`/`params_from_study_gnn`/
`plot_optuna_diagnostics_gnn` from the Run 2 section unchanged - only the
data (`data_run3`/`target_yields_run3`) and the resulting `study_run3`/
`best_params_run3` are new, so the Run 2 study/hyperparameters above stay
untouched.

In [ ]:
study_run3 = run_optuna_search_gnn(data_run3, target_yields_run3, study_name="gnn_opt_run3")
best_params_run3 = params_from_study_gnn(study_run3)
plot_optuna_diagnostics_gnn(study_run3, title_suffix="(2l2tau, Run 3, GNN)")

# ---- Refit the fold-0 preview model with the tuned hyperparameters, so the
# rest of this track's (feature-importance/physics-FOM) cells below reflect
# the tuned architecture, not the arbitrary defaults.
model_run3, history_run3, best_val_auc_run3, best_train_auc_run3, best_train_auc_eval_run3 = train_model(
    fd0_run3["X_train_t"], fd0_run3["y_train_t"], fd0_run3["w_train_fit_t"], fd0_run3["w_train_abs_t"],
    fd0_run3["X_val_t"], fd0_run3["y_val_t"], fd0_run3["w_val_abs_t"],
    hidden_channels=best_params_run3["hidden_channels"], dropout=best_params_run3["dropout"],
    lr=best_params_run3["lr"], batch_size=best_params_run3["batch_size"],
    patience=best_params_run3["patience"], verbose=False,
)
print(f"\nTuned fold-0 preview (Run 3): val_auc={best_val_auc_run3:.4f} | "
      f"hidden_channels={best_params_run3['hidden_channels']} | batch_size={best_params_run3['batch_size']}")

## Evaluation — Run 3

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_run3["train_loss"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[0].plot(history_run3["train_loss_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[0].plot(history_run3["val_loss"], label="val", color="tab:orange")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("weighted BCE loss")
axes[0].set_title("Loss (Run 3)")
axes[0].legend()

axes[1].plot(history_run3["train_auc"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[1].plot(history_run3["train_auc_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[1].plot(history_run3["val_auc"], label="val", color="tab:orange")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("weighted AUC")
axes[1].set_title("AUC (Run 3)")
axes[1].legend()

plt.tight_layout()
plt.show()

y_val_scores_run3 = score_dataset(model_run3, fd0_run3["X_val_t"]).cpu().numpy()
y_val_run3 = fd0_run3["y_val_t"].cpu().numpy()
w_val_run3 = fd0_run3["w_val_abs_t"].cpu().numpy()

auc_val_run3 = roc_auc_score(y_val_run3, y_val_scores_run3, sample_weight=w_val_run3)
print(f"Final weighted val AUC (Run 3, fold-0 preview) = {auc_val_run3:.4f}")

fpr_run3, tpr_run3, _ = roc_curve(y_val_run3, y_val_scores_run3, sample_weight=w_val_run3)
plt.figure(figsize=(5, 5))
plt.plot(fpr_run3, tpr_run3, label=f"GNN (AUC = {auc_val_run3:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve (validation) — Run 3, fold-0 preview")
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance (Permutation) — Run 3

Reuses `permutation_importance_gnn`/`score_node_features`/
`plot_importance_bar` from the Run 2 section unchanged.

In [ ]:
importance_run3 = permutation_importance_gnn(
    model_run3, fd0_run3["val_scaled"], fd0_run3["val_imp"], y_val_run3, w_val_run3, REQUIRED_OBJECT_COLUMNS, n_repeats=5,
)
plot_importance_bar(
    importance_run3, top_n=30,
    title=f"GNN permutation importance — {len(REQUIRED_OBJECT_COLUMNS)} per-object input columns (2l2tau, Run 3, fold-0 preview)",
)
importance_run3

## Physics Figure of Merit — Run 3

Reuses `significance_scan` from the K-Fold helper library. The score cut is
selected by scanning Run 3's own fold-0-preview VAL only.

In [ ]:
w_val_signed_run3 = fd0_run3["val_df"]["w_phys"].to_numpy()
z_val_run3, thr_val_run3 = significance_scan(y_val_run3, y_val_scores_run3, w_val_signed_run3)

print(f"Weighted val AUC (Run 3, fold-0 preview) = {auc_val_run3:.4f}")
print(f"Max Asimov Z (val, Run 3, fold-0 preview) = {z_val_run3:.3f} at score cut = {thr_val_run3:.4f}")

### K-Fold Production Run & Artifacts (Run 3)

Same convention as the Run 2 section: `run_kfold_gnn` trains one `ObjectGNN`
per outer fold (frozen default hyperparameters) on `data_run3` and scores
every event with a model that never trained on it, then the fold-0-preview
VAL-selected cut (`thr_val_run3`) is applied FROZEN to the OOF scores.

In [ ]:
# ---- K-FOLD PRODUCTION RUN (Run 3) -----------------------------------------
oof_df_run3, models_run3, scalers_run3, medians_run3 = run_kfold_gnn(
    data_run3, target_yields_run3, cell_cols=(), n_folds=N_FOLDS, label="Run3",
    hidden_channels=best_params_run3["hidden_channels"], dropout=best_params_run3["dropout"],
    lr=best_params_run3["lr"], batch_size=best_params_run3["batch_size"],
    patience=best_params_run3["patience"],
)

oof_auc_run3 = plot_oof_roc(oof_df_run3, title="2l2tau Run 3 (GNN) - OOF ROC")

save_track_artifacts_gnn(models_run3, scalers_run3, medians_run3, oof_df_run3, BASE_DIR_RUN3, "run3", hyperparams=best_params_run3)

sel_test_run3 = oof_df_run3["score"] >= thr_val_run3
S_test_run3 = oof_df_run3.loc[sel_test_run3 & (oof_df_run3["label"] == 1), "w_phys"].sum()
B_test_run3 = oof_df_run3.loc[sel_test_run3 & (oof_df_run3["label"] == 0), "w_phys"].sum()
z_test_run3 = (np.sqrt(2 * ((S_test_run3 + B_test_run3) * np.log(1 + S_test_run3 / B_test_run3) - S_test_run3))
               if (S_test_run3 > 0 and B_test_run3 > 0) else np.nan)

print(f"\nWeighted AUC (Run 3): val (fold-0 preview) = {auc_val_run3:.4f}  |  "
      f"OOF (5-fold, scored once per event) = {oof_auc_run3:.4f}")
print(f"At the fold-0-preview VAL-selected score cut = {thr_val_run3:.4f} (frozen, NOT re-scanned on OOF):")
print(f"  OOF S = {S_test_run3:.2f} | OOF B = {B_test_run3:.2f} | OOF Z = {z_test_run3:.3f}  (val Z was {z_val_run3:.3f})")

## Sanity Checks & Summary — Run 3

In [ ]:
# ---- Determinism check, same acceptance criterion as the Run 2 section:
# re-seeding immediately before each of two short training runs should give
# an EXACT match.

set_seed(RANDOM_STATE)
_, _, det_check_a_run3, _, _ = train_model(
    fd0_run3["X_train_t"], fd0_run3["y_train_t"], fd0_run3["w_train_fit_t"], fd0_run3["w_train_abs_t"],
    fd0_run3["X_val_t"], fd0_run3["y_val_t"], fd0_run3["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)

set_seed(RANDOM_STATE)
_, _, det_check_b_run3, _, _ = train_model(
    fd0_run3["X_train_t"], fd0_run3["y_train_t"], fd0_run3["w_train_fit_t"], fd0_run3["w_train_abs_t"],
    fd0_run3["X_val_t"], fd0_run3["y_val_t"], fd0_run3["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)
assert det_check_a_run3 == det_check_b_run3, f"Determinism check FAILED (Run 3): {det_check_a_run3} != {det_check_b_run3}"

print(f"Determinism check passed (Run 3): two re-seeded runs give identical val_auc = {det_check_a_run3:.6f}")

# ---- Final summary ----------------------------------------------------------

summary_run3 = pd.DataFrame([{
    "model": f"GNN (object nodes, {N_NODES} nodes/event)",
    "val_auc (fold-0 preview)": auc_val_run3,
    "max_asimov_Z (val, fold-0 preview)": z_val_run3,
    "oof_auc (5-fold, scored once per event)": oof_auc_run3,
    "oof_Z_at_val_cut": z_test_run3,
}])

print("\nFinal summary (GNN, 2l2tau Run 3):")
print(summary_run3.to_string(index=False))
summary_run3

## Combined Run 2 + Run 3 (Separate Downstream Section)

Same convention as the Run 3 section above: reuses every function/class from
the Run 2 section unchanged (node schema, `stack_node_features`,
`build_graph_tensors`, `ObjectGNN`/`build_model`, `train_model`/`run_epoch`/
`score_dataset`, `permutation_importance_gnn`/`score_node_features`/
`plot_importance_bar`, `significance_scan`, `assign_folds`,
`prepare_fold_tensors_gnn`, `run_kfold_gnn`, `plot_oof_roc`,
`save_track_artifacts_gnn`) - only the data and everything derived from it
get `_comb`-suffixed variable names. Requires the Run 2 and Run 3 sections
above to have already run in this kernel (`data`/`features` and
`data_run3`/`features_run3` must exist). `run_kfold_gnn` is called with
`cell_cols=("run",)` here (same as `DNN.ipynb`'s Combined track), so
`make_fit_weights` balances signal/background WITHIN each run era first,
before the global mean-normalization - this stops whichever run has more MC
events from dominating the pooled loss.

## Load Combined Data

Same convention as `DNN.ipynb`'s Combined track: concatenate Run 2 and
Run 3 with a `run` tag column, restricted to features common to BOTH runs
(the intersection, not just each run's own leakage-free set), then
fold-assigned the same way as every other track.

In [ ]:
candidate_features_comb = sorted(set(features) & set(features_run3))
keep_cols_comb = candidate_features_comb + ["w_phys", "label", "process", EVENT_ID_BRANCH]

data_run2_tagged = data[keep_cols_comb].copy()
data_run2_tagged["run"] = 2

data_run3_tagged = data_run3[keep_cols_comb].copy()
data_run3_tagged["run"] = 3

data_comb_raw = pd.concat([data_run2_tagged, data_run3_tagged], ignore_index=True)
data_comb, features_comb = clean_data(data_comb_raw, candidate_features_comb)
data_comb = assign_folds(data_comb)

print(f"\nCombined: {len(data_comb)} events, {len(features_comb)} features "
      f"(intersection of Run2's {len(features)} and Run3's {len(features_run3)})")

for run_label in (2, 3):
    sub = data_comb[data_comb["run"] == run_label]
    print(f"Run {run_label}: signal yield = {sub.loc[sub.label==1,'w_phys'].sum():.2f} | "
          f"background yield = {sub.loc[sub.label==0,'w_phys'].sum():.2f} | n_events = {len(sub)}")

print(f"\nFold sizes (fold = eventNumber % {N_FOLDS}), Combined:")
print(data_comb["fold"].value_counts().sort_index())

## Sentinel Audit (-1) — Combined

Same diagnostic as the Run 2/Run 3 sections, run separately on `data_comb`/
`features_comb`.

In [ ]:
NEG1_SENTINEL_FEATURES_comb = set()

neg1_rows_comb = []
for f in features_comb:
    vals = data_comb[f]
    frac_neg1 = (vals == -1).mean()
    if frac_neg1 == 0:
        continue
    above = vals[vals > -1]
    gap = (above.min() - (-1)) if len(above) else np.nan
    neg1_rows_comb.append({"feature": f, "frac_exactly_-1": frac_neg1, "gap_to_next_value_above": gap})

neg1_df_comb = pd.DataFrame(neg1_rows_comb).sort_values("frac_exactly_-1", ascending=False)
print(f"{len(neg1_df_comb)} / {len(features_comb)} features have at least one row exactly equal to -1:")
print(neg1_df_comb.to_string(index=False))

for f in NEG1_SENTINEL_FEATURES_comb:
    data_comb[f] = data_comb[f].mask(data_comb[f] == -1)

if NEG1_SENTINEL_FEATURES_comb:
    print(f"\nMasked -1 -> NaN for: {sorted(NEG1_SENTINEL_FEATURES_comb)}")
else:
    print("\nNEG1_SENTINEL_FEATURES_comb is empty - no -1 values masked.")

## Fold-0 Preview: Preprocessing & Weight-Balance Diagnostics — Combined

The object/node schema is run-independent, reused unchanged from the Run 2
section. `prepare_fold_tensors_gnn` is called with `cell_cols=("run",)` so
the fold-0 preview's training weights are balanced within each run era
first, same convention as the K-Fold Production Run below.

In [ ]:
missing_comb = [c for c in REQUIRED_OBJECT_COLUMNS if c not in features_comb]
assert not missing_comb, f"Node schema references columns dropped by clean_data / the leakage policy (Combined): {missing_comb}"

BASE_DIR_COMB = Path("PPSSP_2026/2l2tau/combined")
PLOTS_DIR_COMB = BASE_DIR_COMB / "plots"
PLOTS_DIR_COMB.mkdir(parents=True, exist_ok=True)

target_yields_comb = compute_process_yield_targets(data_comb)

fd0_comb = prepare_fold_tensors_gnn(data_comb, target_yields_comb, cell_cols=("run",), n_folds=N_FOLDS, k=0)

w_before_comb = fd0_comb["train_df"]["w_phys"].to_numpy()
w_after_comb = fd0_comb["w_train_fit"]
y_preview_comb = fd0_comb["train_df"]["label"].to_numpy()

plot_weight_balance(
    y_preview_comb, w_before_comb, w_after_comb,
    title="Combined (GNN) - make_fit_weights balancing (fold 0 train, per-run cell)",
    save_path=PLOTS_DIR_COMB / "CombinedWeightBalance_GNN.png",
)

print(f"Built {fd0_comb['X_train_t'].shape[0]} train / {fd0_comb['X_val_t'].shape[0]} val / "
      f"{fd0_comb['X_test_t'].shape[0]} test graphs (Combined, fold 0 preview)")

print("\nN_eff (training sample, positive-only, post yield-rescale) by run x label - fold 0:")
print(n_eff_table(fd0_comb["train_df"], ["run", "label"]))
print("\nN_eff (eval sample, signed, FULL fold-0 test partition) by run x label:")
print(n_eff_table(fd0_comb["test_df"], ["run", "label"]))
print(f"\nDropped {fd0_comb['n_dropped_train']} negative-w_phys training rows in this "
      f"preview fold (kept, not abs'd, elsewhere - see prepare_fold_data).")

## Training Loop — Combined

Reuses `train_model`/`run_epoch`/`build_model`/`ObjectGNN` from the Run 2
section unchanged, with explicit `fd0_comb` tensors passed in so the Run 2
and Run 3 models/histories are untouched.

In [ ]:
model_comb, history_comb, best_val_auc_comb, best_train_auc_comb, best_train_auc_eval_comb = train_model(
    fd0_comb["X_train_t"], fd0_comb["y_train_t"], fd0_comb["w_train_fit_t"], fd0_comb["w_train_abs_t"],
    fd0_comb["X_val_t"], fd0_comb["y_val_t"], fd0_comb["w_val_abs_t"],
)

print(f"\nBest val_auc = {best_val_auc_comb:.4f} | train_auc (dropout on) = {best_train_auc_comb:.4f} "
      f"| train_auc_eval (dropout off, comparable) = {best_train_auc_eval_comb:.4f}")

## Optuna Hyperparameter Search — Combined

Reuses `train_model_for_search`/`run_optuna_search_gnn`/`params_from_study_gnn`/
`plot_optuna_diagnostics_gnn` from the Run 2 section unchanged - only the
data (`data_comb`/`target_yields_comb`) and the resulting `study_comb`/
`best_params_comb` are new. `cell_cols=("run",)` is passed through to
`run_optuna_search_gnn` (and its internal `prepare_fold_tensors_gnn` calls),
same convention as the Combined track's other per-run weight balancing.

In [ ]:
study_comb = run_optuna_search_gnn(data_comb, target_yields_comb, cell_cols=("run",), study_name="gnn_opt_combined")
best_params_comb = params_from_study_gnn(study_comb)
plot_optuna_diagnostics_gnn(study_comb, title_suffix="(2l2tau, Combined, GNN)")

# ---- Refit the fold-0 preview model with the tuned hyperparameters, so the
# rest of this track's (feature-importance/physics-FOM) cells below reflect
# the tuned architecture, not the arbitrary defaults.
model_comb, history_comb, best_val_auc_comb, best_train_auc_comb, best_train_auc_eval_comb = train_model(
    fd0_comb["X_train_t"], fd0_comb["y_train_t"], fd0_comb["w_train_fit_t"], fd0_comb["w_train_abs_t"],
    fd0_comb["X_val_t"], fd0_comb["y_val_t"], fd0_comb["w_val_abs_t"],
    hidden_channels=best_params_comb["hidden_channels"], dropout=best_params_comb["dropout"],
    lr=best_params_comb["lr"], batch_size=best_params_comb["batch_size"],
    patience=best_params_comb["patience"], verbose=False,
)
print(f"\nTuned fold-0 preview (Combined): val_auc={best_val_auc_comb:.4f} | "
      f"hidden_channels={best_params_comb['hidden_channels']} | batch_size={best_params_comb['batch_size']}")

## Evaluation — Combined

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_comb["train_loss"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[0].plot(history_comb["train_loss_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[0].plot(history_comb["val_loss"], label="val", color="tab:orange")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("weighted BCE loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history_comb["train_auc"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[1].plot(history_comb["train_auc_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[1].plot(history_comb["val_auc"], label="val", color="tab:orange")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("weighted AUC")
axes[1].set_title("AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

y_val_scores_comb = score_dataset(model_comb, fd0_comb["X_val_t"]).cpu().numpy()
y_val_comb = fd0_comb["y_val_t"].cpu().numpy()
w_val_comb = fd0_comb["w_val_abs_t"].cpu().numpy()

auc_val_comb = roc_auc_score(y_val_comb, y_val_scores_comb, sample_weight=w_val_comb)
print(f"Final weighted val AUC (Combined, fold-0 preview) = {auc_val_comb:.4f}")

fpr_comb, tpr_comb, _ = roc_curve(y_val_comb, y_val_scores_comb, sample_weight=w_val_comb)
plt.figure(figsize=(5, 5))
plt.plot(fpr_comb, tpr_comb, label=f"GNN (AUC = {auc_val_comb:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve (validation, Combined, fold-0 preview)")
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance (Permutation) — Combined

Reuses `permutation_importance_gnn`/`score_node_features`/`plot_importance_bar`
from the Run 2 section unchanged.

In [ ]:
importance_comb = permutation_importance_gnn(
    model_comb, fd0_comb["val_scaled"], fd0_comb["val_imp"], y_val_comb, w_val_comb, REQUIRED_OBJECT_COLUMNS, n_repeats=5,
)
plot_importance_bar(
    importance_comb, top_n=30,
    title=f"GNN permutation importance — {len(REQUIRED_OBJECT_COLUMNS)} per-object input columns (2l2tau, Combined, fold-0 preview)",
)
importance_comb

## Physics Figure of Merit — Combined

Reuses `significance_scan` from the K-Fold helper library. The score cut is
selected by scanning the Combined fold-0-preview VAL split only.

In [ ]:
w_val_signed_comb = fd0_comb["val_df"]["w_phys"].to_numpy()
z_val_comb, thr_val_comb = significance_scan(y_val_comb, y_val_scores_comb, w_val_signed_comb)

print(f"Weighted val AUC (Combined, fold-0 preview) = {auc_val_comb:.4f}")
print(f"Max Asimov Z (val, Combined, fold-0 preview) = {z_val_comb:.3f} at score cut = {thr_val_comb:.4f}")

### K-Fold Production Run & Artifacts (Combined)

Same convention as the Run 2/Run 3 sections, with `cell_cols=("run",)`
passed to `run_kfold_gnn` so each fold's training weights are balanced
within each run era before the global mean-normalization (same as
`DNN.ipynb`'s Combined track).

In [ ]:
# ---- K-FOLD PRODUCTION RUN (Combined) --------------------------------------
oof_df_comb, models_comb, scalers_comb, medians_comb = run_kfold_gnn(
    data_comb, target_yields_comb, cell_cols=("run",), n_folds=N_FOLDS, label="Combined",
    hidden_channels=best_params_comb["hidden_channels"], dropout=best_params_comb["dropout"],
    lr=best_params_comb["lr"], batch_size=best_params_comb["batch_size"],
    patience=best_params_comb["patience"],
)

oof_auc_comb = plot_oof_roc(oof_df_comb, title="2l2tau Combined Run2+Run3 (GNN) - OOF ROC")

save_track_artifacts_gnn(models_comb, scalers_comb, medians_comb, oof_df_comb, BASE_DIR_COMB, "combined", hyperparams=best_params_comb)

sel_test_comb = oof_df_comb["score"] >= thr_val_comb
S_test_comb = oof_df_comb.loc[sel_test_comb & (oof_df_comb["label"] == 1), "w_phys"].sum()
B_test_comb = oof_df_comb.loc[sel_test_comb & (oof_df_comb["label"] == 0), "w_phys"].sum()
z_test_comb = (np.sqrt(2 * ((S_test_comb + B_test_comb) * np.log(1 + S_test_comb / B_test_comb) - S_test_comb))
               if (S_test_comb > 0 and B_test_comb > 0) else np.nan)

print(f"\nWeighted AUC (Combined): val (fold-0 preview) = {auc_val_comb:.4f}  |  "
      f"OOF (5-fold, scored once per event) = {oof_auc_comb:.4f}")
print(f"At the fold-0-preview VAL-selected score cut = {thr_val_comb:.4f} (frozen, NOT re-scanned on OOF):")
print(f"  OOF S = {S_test_comb:.2f} | OOF B = {B_test_comb:.2f} | OOF Z = {z_test_comb:.3f}  (val Z was {z_val_comb:.3f})")

## Sanity Checks & Summary — Combined

In [ ]:
set_seed(RANDOM_STATE)
_, _, det_check_a_comb, _, _ = train_model(
    fd0_comb["X_train_t"], fd0_comb["y_train_t"], fd0_comb["w_train_fit_t"], fd0_comb["w_train_abs_t"],
    fd0_comb["X_val_t"], fd0_comb["y_val_t"], fd0_comb["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)

set_seed(RANDOM_STATE)
_, _, det_check_b_comb, _, _ = train_model(
    fd0_comb["X_train_t"], fd0_comb["y_train_t"], fd0_comb["w_train_fit_t"], fd0_comb["w_train_abs_t"],
    fd0_comb["X_val_t"], fd0_comb["y_val_t"], fd0_comb["w_val_abs_t"],
    hidden_channels=16, n_epochs=5, patience=5, verbose=False,
)
assert det_check_a_comb == det_check_b_comb, f"Determinism check FAILED (Combined): {det_check_a_comb} != {det_check_b_comb}"

print(f"Determinism check passed (Combined): two re-seeded runs give identical val_auc = {det_check_a_comb:.6f}")

summary_comb = pd.DataFrame([{
    "model": f"GNN (object nodes, {N_NODES} nodes/event)",
    "val_auc (fold-0 preview)": auc_val_comb,
    "max_asimov_Z (val, fold-0 preview)": z_val_comb,
    "oof_auc (5-fold, scored once per event)": oof_auc_comb,
    "oof_Z_at_val_cut": z_test_comb,
}])

print("\nFinal summary (GNN, 2l2tau Combined Run2+Run3):")
print(summary_comb.to_string(index=False))
summary_comb